## 20 — CD e rollback do serving (preparação)

Primeiro ensaio blue-green: duas identidades de release, **mesmos pesos e mesma imagem fixada**, gateway NGINX e recursos temporários próprios. O pacote corrompido deve ser bloqueado antes de receber tráfego; B precisa passar readiness/identidade/paridade; uma falha explícita deve produzir 502 e acionar retorno verificado a A. Sem MLflow/S3/datasets ou retreino. **Gate Docker nativo pendente.** Não é publicação automática nem prova de zero downtime.

Ver [procedimento](../../docs/operations/SERVING_ROLLOUT.md).


## Gate explícito no host

O roteiro completo está no [runbook](../../docs/operations/SERVING_ROLLOUT.md). Requer a release já restaurada e a imagem Docker anterior ainda presente. O ensaio cria recursos próprios; não altera o serving em uso nem depende de tracking ativo.

```bash
docker pull nginx:1.30.6-alpine@sha256:970364228b1e4f932abc22d321932816782cca8a20b85a77552a7cfe9acfb3b5
.venv/bin/python -m tox -e serving-rollout
```

O recibo fica em `data/serving/rollouts/<uuid>/result.json`; somente status success com sequência completa, fontes intactas e cleanup concluído aprova o ensaio. Em falha, preservar o recibo. Não repetir treino/holdout/exportação.


### Correção da rede do ensaio de rollout — 2026-10-19

A primeira execução encontrou o backend A saudável, mas o gate HTTP do host expirou antes do roteamento. O cleanup foi confirmado. A e B passam a ficar sem portas publicadas na rede interna; os gates fazem HTTP dentro dos containers e preservam identidade, score e HTTP 422. Só o NGINX publica loopback por uma bridge adicional de entrada. A validação Docker da correção ainda está pendente.


## Blue-green: aceite nativo e decisão — 2026-10-19

**O gate descrito como pendente nas células de preparação está concluído.** O autor aprovou a troca A → B → A pela mesma URL NGINX, com bloqueio do pacote inválido antes da troca, falha de B observada como HTTP 502 e retorno verificado em **3,10 s**. B e o retorno reproduziram o score **0.4178630794049542**. Pesos, imagem e bytes de entrada preservados; recursos temporários removidos.

Validação informada: **110 testes de serving aprovados e 2 opt-in ignorados** (pytest 4,28 s; tox 6,94 s); teste Docker **1 aprovado em 21,42 s** (tox 23,90 s). Os hashes do controller/checker e os pins foram conciliados com o código/evidência; o assistente não acessou o engine ou o recibo nativo independentemente.

A correção separou os backends privados da entrada do gateway e foi aprovada nessa execução. Mantemos **blue-green** para o portfolio: planejar uma réplica ativa, candidata durante a entrega e anterior disponível por uma janela de rollback a definir. Essa política permanente ainda não existe: hoje o ensaio remove todos os seus recursos. Canary depende de tráfego e observação representativos.

Próximo incremento: conectar imagem/release fixadas, gates, promoção explícita e rollback ao fluxo versionado de entrega; depois provisionamento KinD + Helm. Este gate não valida novo modelo/imagem, CD de produção, zero downtime ou SLA; 3,10 s é uma observação local.

[Procedimento e limites](../../docs/operations/SERVING_ROLLOUT.md#aceite-nativo--2026-10-19) · [Evidência do autor](../../references/evidence/serving_rollout_author_validation_2026-10-19.json). Não é necessário repetir o ensaio para aplicar este fechamento documental.


## Entrega versionada: build e gates na CI — preparação de 2026-10-19

Após o aceite da troca/rollback da referência, a CI passa a encadear qualidade → build único → gates HTTP e blue-green sintéticos → exportação da mesma imagem. O descritor vincula commit, hashes das entradas, ID local da imagem, release sintética e recibos; falha de gate ou cleanup impede descritor qualificado. Todos os resultados ficam em `data/serving/deliveries/`, com artifact da CI retido por três dias. O ID local é separado de digest de registry; não há publicação ou deploy no laptop.

A pequena fixture é ajustada sobre dados gerados; a referência histórica não é lida ou retreinada. A verificação offline do pacote compara bytes/recibos, sem carregar imagem, desserializar modelo ou validar assinatura. O build e a execução completa Docker/CI deste novo fluxo **ainda não foram aprovados**. Isso não altera o aceite anterior, limitado aos pins nele registrados.

Depois do primeiro aceite, validar consumo do arquivo em outro alvo e promoção explícita de release aprovada. KinD/Helm e política de retenção da versão anterior mantêm critérios próprios. [Procedimento e limites](../../docs/operations/SERVING_ROLLOUT.md#entrega-versionada-na-ci--implementação-de-2026-10-19).


### Fixture de entrega: permissão para o UID 10001 — 2026-10-19

Na primeira execução nativa, o build terminou, mas o gate HTTP falhou na inicialização: PermissionError em `/model/manifest.json`. O gravador atômico publica JSONs 0600; o controller não havia preparado a fixture para outro UID. A correção valida hashes e evidência sintética, normaliza apenas a fixture nova (diretório 0755/arquivos 0644) e preserva o job privado, os bytes e a origem. Usuário 10001 e mount read-only continuam iguais.

Novos testes exigem as permissões antes do checker sob umask 022/077 e rejeitam normalização de release histórica. O acesso real pelo container ainda depende do próximo gate nativo; a tentativa falha é preservada. [Diagnóstico e correção](../../docs/operations/SERVING_ROLLOUT.md#correção-das-permissões-da-fixture--2026-10-19).


## Entrega versionada: aceite nativo — 2026-10-19

**O gate local descrito como pendente nas células de preparação está concluído.** O autor aprovou `serving-delivery`: 1 teste em **62,04 s**, tox em **64,51 s**. Build da imagem do commit `0b95136a462583441044d50de0884a558777b260`, HTTP e blue-green/rollback com fixture sintética, exportação da mesma imagem e verificação do pacote passaram. A correção de leitura pelo UID 10001 foi validada no fluxo completo. Foram removidos os recursos temporários e a tag própria.

O descritor informado tem SHA-256 `aa0e79886800765c516d3ed776295dc46d657a7ea50eb7c88900241d8c0ff0d7`; a imagem local tem ID `sha256:bc2ba70a90de114613bd8679e79ccfbb7a17de048f72f16323ca26713e1dae8d`. O pacote inventaria **11 arquivos**, com **175.020.032 bytes** em image.tar. O autor executou a conferência do hash do descritor antes de enviar o resumo; o assistente não recebeu os bytes do descritor, inventário completo ou arquivo de imagem para verificação independente.

A referência histórica não foi acessada ou retreinada; houve apenas fit da fixture gerada. Sem publicação em registry, promoção ou serviço permanente. GitHub Actions/upload remoto e recarga de image.tar em outro alvo ainda não foram aprovados. O próximo gate consome o arquivo verificado, sem rebuild/fit, e confirma identidade e HTTP antes de qualquer promoção. Não é necessário repetir build ou teste para este fechamento.

[Procedimento e limites](../../docs/operations/SERVING_ROLLOUT.md#aceite-nativo-da-entrega-versionada--2026-10-19) · [Evidência do autor](../../references/evidence/serving_delivery_author_validation_2026-10-19.json).


## Consumir a imagem aprovada e transferir a entrega entre jobs — preparação de 2026-10-19

O novo consumidor exige commit/hash esperados antes de carregar `image.tar`, verifica a identidade da imagem e executa o checker HTTP em container novo, sem rebuild ou fit. Copia somente a release sintética para o mount e preserva os bytes/permissões da entrega original; remove apenas seus containers/tag importada. O recibo de consumo é separado do descritor do produtor.

A CI encadeia qualidade → produtor → consumidor em outro runner, usando o ID do artifact e o hash publicados pelo produtor. O gate local consome o pacote já aprovado; pode compartilhar cache/engine e não comprova engine independente. Aceites nativo e Actions **pendentes**; sem promoção, dados históricos ou deploy permanente. [Procedimento e limites](../../docs/operations/SERVING_ROLLOUT.md#consumo-da-imagem-arquivada-e-ci-entre-jobs--2026-10-19).


## Aceite nativo do consumo da imagem — 2026-10-19

O autor aprovou serving-consumer: 1 teste em 32,41 s, tox OK em 44,93 s. O arquivo da entrega já aprovada foi carregado e executado em container novo; imagem/release/modelo/HTTP conferidos, 422 observado, origem preservada e recursos próprios removidos. Score sintético: 0.03463335444035967. Sem rebuild, fit ou referência histórica.

O ID da imagem estava ausente antes do load; isso não prova camadas ausentes, cache vazio ou engine independente. Aceite recebido pelo chat, sem acesso independente do assistente ao daemon/arquivo. Gates nativos encerrados; próximo passo é commit/push para validar quality → serving-delivery → serving-consumer no GitHub Actions, cujo aceite ainda está pendente. [Resultado e limites](../../docs/operations/SERVING_ROLLOUT.md#aceite-nativo-do-consumo-da-imagem--2026-10-19).
